# CottonLens — 3 günlük tabular sprint
T4 GPU seç. Kaynaklar Drive'a doğrudan kopyalanır. R2 verileri ve sonuçları korunur.
İlk çalıştırmada 1–6 kod hücrelerini sırayla çalıştır; 6. hücre yalnız ilk altı tarifi/horizon çalıştırır.
Sonraki aşamalar: search → refine → cadence → lock → benchmark → reproduce → release.
Her aşamadan sonra sonuç dosyası Drive'da kalır. Yeni source veya veri ile aynı experiment kullanılmaz.
48 saatlik arama süresi 5. hücrede başlar. Süre dolarsa yeni arama yerine tamamlanan sonuçlarla lock uygulanır.
Gerçek training yalnız Colab'da. 2024+ audit arama veya seçimde kullanılmaz.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'tabular-sprint-v1'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-sprint-source-v1.zip'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected_zip, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'], 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup()  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
SMOKE_PASSED = False
SMOKE_REPORT = DRIVE_ROOT / 'reports' / ('sprint-smoke-' + identity + '.json')
if SMOKE_REPORT.exists():
    saved = json.loads(SMOKE_REPORT.read_text())
    assert saved['status'] == 'passed' and saved['source_id'] == identity, 'Existing smoke failed; inspect it before retrying'
else:
    run([PYTHON, '-m', 'cottonlens_ml.smoke', '--repo', REPO,
         '--report', SMOKE_REPORT, '--inference-python', INFERENCE / 'bin/python'])
SMOKE_PASSED = True
print('Smoke passed:', SMOKE_REPORT)

In [ ]:
assert SMOKE_PASSED
BASE = [PYTHON, '-m', 'cottonlens_ml.sprint', '--repo', REPO,
        '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
run([*BASE, '--stage', 'prepare', '--smoke-report', SMOKE_REPORT])
SPRINT_ROOT = DRIVE_ROOT / 'experiments' / EXPERIMENT
print('Frozen sprint:', SPRINT_ROOT)

In [ ]:
STAGE = 'search'  # Later: refine, cadence, lock, benchmark, reproduce, release
assert STAGE in ('search', 'refine', 'cadence', 'lock', 'benchmark', 'reproduce', 'release', 'report')
run([*BASE, '--stage', STAGE])
reports = sorted(SPRINT_ROOT.glob('summary-*.txt'), key=lambda p: p.stat().st_mtime)
if reports:
    print(reports[-1].read_text())
    print('Send this report:', reports[-1])
    from google.colab import files
    files.download(str(reports[-1]))

In [ ]:
# Run only after the release stage. This does not touch the R2 latest pointer.
pending = json.loads((SPRINT_ROOT / 'release.json').read_text())
RELEASE = Path(pending['path'])
run([PYTHON, '-m', 'cottonlens_ml.validate_release', '--repo', REPO,
     '--drive-root', SPRINT_ROOT, '--release', RELEASE,
     '--inference-python', INFERENCE / 'bin/python'])
run([PYTHON, '-m', 'cottonlens_ml.report', '--drive-root', SPRINT_ROOT])
print('Validated release:', RELEASE)